In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("ai_student_impact_dataset.csv")
y = df["Burnout_Risk_Level"]
X = df.drop(columns="Burnout_Risk_Level")


In [3]:

from sklearn.model_selection import train_test_split

# Separizziamo X (le feature) e y (il target, es. 'Burnout_Risk_Level')
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,          # Scegliamo il 20% dei dati per il test set e l'80% per il train   
    random_state=42,        # Inseriamo un seme casuale fisso per rendere i risultati riproducibili
    stratify=y              # FONDAMENTALE: Mantiene le stesse proporzioni delle classi del target sia in train che in test
)

target_map = {'Low': 0, 'Medium': 1, 'High': 2}   # random forest e xgboost lavorano meglio se y è mappata
y_train = y_train.map(target_map)
y_test = y_test.map(target_map)

In [4]:
X_train.drop(columns=['Student_ID'], inplace=True, errors='ignore')
X_test.drop(columns=['Student_ID'], inplace=True, errors='ignore') # colonna inutile

mappa_skills = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}   #lable encoding risponde alla presenza di un ordinamento
X_train['Prompt_Engineering_Skill_Num'] = X_train['Prompt_Engineering_Skill'].map(mappa_skills)
X_test['Prompt_Engineering_Skill_Num'] = X_test['Prompt_Engineering_Skill'].map(mappa_skills)

X_train = X_train.drop('Prompt_Engineering_Skill', axis=1)
X_test = X_test.drop('Prompt_Engineering_Skill', axis=1)

In [5]:
X_train = pd.get_dummies(X_train, columns=['Primary_Use_Case'], dtype=int)   
X_test = pd.get_dummies(X_test, columns=['Primary_Use_Case'], dtype=int) 

# one hot encoding, non c'era ordinamento o relazione diretta con la y però 
# interagisce con altre feature quindi le conservo per un eventuale feature engeneering 

# e comunque... si fa sempre in tempo ad eliminare

In [6]:
X_train = pd.get_dummies(X_train, columns=['Major_Category'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['Major_Category'], dtype=int)

X_train = pd.get_dummies(X_train, columns=['Institutional_Policy'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['Institutional_Policy'], dtype=int)

# presenta un legame indiretto con "livelli di burnout" attraverso "l'ansia da esame "

In [13]:
X_test.info()

<class 'pandas.DataFrame'>
Index: 10000 entries, 18437 to 43395
Data columns (total 24 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Year_of_Study                               10000 non-null  str    
 1   Pre_Semester_GPA                            10000 non-null  float64
 2   Weekly_GenAI_Hours                          10000 non-null  float64
 3   Tool_Diversity                              10000 non-null  int64  
 4   Paid_Subscription                           10000 non-null  bool   
 5   Traditional_Study_Hours                     10000 non-null  float64
 6   Perceived_AI_Dependency                     10000 non-null  int64  
 7   Anxiety_Level_During_Exams                  10000 non-null  int64  
 8   Post_Semester_GPA                           10000 non-null  float64
 9   Skill_Retention_Score                       10000 non-null  float64
 10  Prompt_Engineering_Ski

scelgo di fare target encoding sulla colonna "Year_of_Study", in funzione delle osservazioni in fase di analisi data.
RECAP da analisi dati:
I Freshman (1) iniziano con un livello di burnout alto (il rosso sfiora il 28%). Questo è fisiologico e descrive lo shock da inserimento universitario, le prime sessioni d'esame e il cambio di metodo di studio.

Nei Sophomore (2) e ancora di più nei Junior (3), la fascia rossa si restringe visibilmente, toccando il punto più basso (sotto il 20%). Parallelamente, nei Junior la fascia verde (Low) raggiunge la sua massima espansione (oltre il 40%).

La fascia rossa (High) torna a espandersi, superando nuovamente il 20%.

In [7]:
# 1. Creiamo una copia temporanea del train per calcolare le frequenze
df_temp = X_train.copy()
df_temp['target_temp'] = y_train

# 2. Calcoliamo la tabella delle probabilità per ogni anno di studio
prob_table = pd.crosstab(df_temp['Year_of_Study'], df_temp['target_temp'], normalize='index')
prob_table.columns = [f'Year_of_Study_Prob_{classe}' for classe in prob_table.columns]

# 3. Mappiamo le probabilità creando i nuovi DataFrame codificati
X_train_encoded = X_train.merge(prob_table, on='Year_of_Study', how='left').set_index(X_train.index)
X_test_encoded = X_test.merge(prob_table, on='Year_of_Study', how='left').set_index(X_test.index)

# 4. Gestione di eventuali valori mancanti nel Test Set
for col in prob_table.columns:
    # Rimuoviamo il prefisso per trovare il nome originale della classe (es. 0, 1, 2 oppure 'Low', 'Medium', 'High')
    nome_classe_originale = col.replace('Year_of_Study_Prob_', '')
    
    # Se le classi in y_train sono diventate numeriche (0, 1, 2) a causa di un mapping precedente,
    # convertiamo il nome della classe in intero se è composto da sole cifre
    if str(nome_classe_originale).isdigit():
        nome_classe_originale = int(nome_classe_originale)
        
    prob_globale = (y_train == nome_classe_originale).mean()
    X_test_encoded[col] = X_test_encoded[col].fillna(prob_globale)

# ==============================================================================
# 5. PASSO FONDAMENTALE: AGGIORNARE LE VARIABILI ORIGINALI E RIMUOVERE LA VECCHIA COLONNA
# ==============================================================================
# Sovrascriviamo X_train e X_test con i nuovi DataFrame che contengono le probabilità
X_train = X_train_encoded.copy()
X_test = X_test_encoded.copy()


In [8]:
# 5. Eliminiamo la vecchia colonna categorica 'Year_of_Study' se presente
if 'Year_of_Study' in X_train.columns:
    X_train.drop(columns=['Year_of_Study'], inplace=True)
if 'Year_of_Study' in X_test.columns:
    X_test.drop(columns=['Year_of_Study'], inplace=True)

# Verifica finale per essere sicuri al 100%
print("Colonne create con successo:")
print([c for c in X_train.columns if 'Year_of_Study_Prob' in c])

Colonne create con successo:
['Year_of_Study_Prob_0', 'Year_of_Study_Prob_1', 'Year_of_Study_Prob_2']


In [9]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. La PCA è sensibile alle scale, quindi standardizziamo prima le feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Inizializziamo la PCA chiedendo di mantenere, ad esempio, solo 3 componenti principali
pca = PCA(n_components=3, random_state=42)

# 3. Trasformiamo i nostri dataset enormi in dataset compressi a 3 colonne
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# [Opzionale] Controlliamo quanta informazione abbiamo salvato
print(f"Informazione totale preservata: {sum(pca.explained_variance_ratio_):.2%}")


Informazione totale preservata: 26.34%


**fine preparazione colonne**

inizio addestramento del modello (non ho scalato i valori perchè userò un modello di random forest)

# piano: valutare random forest e xgboost  
fase 1) trovare gli iperparametri migliori per  random forest   (**fatto**)

fase 2) trovare gli iperparametri migliori per  xgboost 

fase 3) addestrare i modelli e confrontarli 

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Inizializziamo il modello base
# Usiamo random_state per garantire la riproducibilità degli split interni della cross-validation
rf_base = RandomForestClassifier(random_state=42)

# 2. Definiamo la griglia degli iperparametri da testare
# NOTA: Puoi stringere o allargare questi range in base alla potenza di calcolo della tua macchina
param_grid = {
    'n_estimators': [80, 100, 150 ],          
    'max_depth': [5,8,10 ],          # Massima profondità di ogni albero (None = cresce al massimo)
    'min_samples_split': [2, 5],          # Numero minimo di campioni richiesti per sdoppiare un nodo
    'min_samples_leaf': [4],            # Numero minimo di campioni richiesti in una foglia (nodo finale)
    'criterion': ['gini']          # Funzione per misurare la qualità dello split
}

# 3. Configuriama la Grid Search
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,                     
    scoring='accuracy',       # da valutare 'f1_macro' o 'roc_auc' se le tue classi sono molto sbilanciate
    n_jobs=-1,               
    verbose=2                 # Stampa a schermo i progressi 
)

# 4. Avviamo la ricerca sul set di addestramento
grid_search.fit(X_train, y_train)

# ==============================================================================
# RISULTATI DELLA RICERCA
# ==============================================================================

print("\n--- Risultati della Grid Search ---")
print(f"Miglior punteggio ottenuto (CV Score): {grid_search.best_score_:.4f}")
print("Migliori iperparametri trovati:")
for param, value in grid_search.best_params_.items():
    print(f"  -> {param}: {value}")


Fitting 5 folds for each of 27 candidates, totalling 135 fits

--- Risultati della Grid Search ---
Miglior punteggio ottenuto (CV Score): 0.5362
Migliori iperparametri trovati:
  -> criterion: gini
  -> max_depth: 10
  -> min_samples_leaf: 4
  -> min_samples_split: 2
  -> n_estimators: 100


param_grid = {
    'n_estimators': [100, 150, 200 ],    

    'max_depth': [ 10, 20, 30],          # Massima profondità di ogni albero (None = cresce al massimo)

    'min_samples_split': [2, 5, 10],          # Numero minimo di campioni richiesti per sdoppiare un nodo

    'min_samples_leaf': [4],            # Numero minimo di campioni richiesti in una foglia (nodo finale)



# addestramento del random forest con gli iperparametri trovati

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Istanziamo il modello con gli iperparametri ottimi trovati
# Aggiungiamo random_state=42 per garantire la riproducibilità dei risultati
best_rf = RandomForestClassifier(
    criterion='gini',
    max_depth=10,
    min_samples_leaf=4,
    min_samples_split=2,
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # Sfrutta tutti i core della CPU per velocizzare l'addestramento
)

# 2. Addestriamo il modello sul set di train (che include il tuo Target Encoding)
best_rf.fit(X_train, y_train)

# 3. Valutiamo le performance sul set di test per verificare che non ci sia overfitting
y_pred = best_rf.predict(X_test)

# Stampiamo i risultati finali
print(f"Accuratezza finale sul Test Set: {accuracy_score(y_test, y_pred):.4f}\n")
print("--- Report di Classificazione Completo ---")
print(classification_report(y_test, y_pred))

Accuratezza finale sul Test Set: 0.5305

--- Report di Classificazione Completo ---
              precision    recall  f1-score   support

           0       0.54      0.47      0.50      3274
           1       0.48      0.62      0.54      4229
           2       0.67      0.45      0.54      2497

    accuracy                           0.53     10000
   macro avg       0.56      0.52      0.53     10000
weighted avg       0.55      0.53      0.53     10000



**Con un'accuratezza del 53.05% su un problema a tre classi (casuale su tre classi = 33%), il modello ha catturato dei pattern utili, ma i dati mostrano che fa fatica a distinguere nettamente tra le classi adiacenti.**

**burnout High: ha una precision molto alta (67%) ma una recall bassa (45%). Questo significa che quando il modello si sbilancia a dire "Rischio Alto", ci indovina quasi sempre, ma si perde per strada più della metà dei veri casi ad alto rischio**



ricorda il map 
'Low': 0, 'Medium': 1, 'High': 2

# faccio un controllo con la PCA

In [12]:


# 1. Istanziamo il modello con gli iperparametri ottimi trovati
# Aggiungiamo random_state=42 per garantire la riproducibilità dei risultati
best_rf_pca = RandomForestClassifier(
    criterion='gini',
    max_depth=10,
    min_samples_leaf=4,
    min_samples_split=2,
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # Sfrutta tutti i core della CPU per velocizzare l'addestramento
)

# 2. Addestriamo il modello sul set di train (che include il tuo Target Encoding)
best_rf_pca.fit(X_train_pca, y_train)

# 3. Valutiamo le performance sul set di test per verificare che non ci sia overfitting
y_pred_pca = best_rf_pca.predict(X_test_pca)

# Stampiamo i risultati finali
print(f"(PCA) Accuratezza finale sul Test Set: {accuracy_score(y_test, y_pred_pca):.4f}\n")
print("--- Report di Classificazione Completo ---")
print(classification_report(y_test, y_pred_pca))

(PCA) Accuratezza finale sul Test Set: 0.5057

--- Report di Classificazione Completo ---
              precision    recall  f1-score   support

           0       0.52      0.40      0.46      3274
           1       0.47      0.66      0.55      4229
           2       0.63      0.37      0.47      2497

    accuracy                           0.51     10000
   macro avg       0.54      0.48      0.49     10000
weighted avg       0.53      0.51      0.50     10000




           0       0.54      0.47      0.50      3274
           1       0.48      0.62      0.54      4229
           2       0.67      0.45      0.54      2497

    accuracy                           0.53     10000
   macro avg       0.56      0.52      0.53     10000
weighted avg       0.55      0.53      0.53     10000

# --------
# da fare ....

In [16]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Poiché abbiamo un problema a 3 classi (Low, Medium, High), impostiamo l'objective corretto
xgb_base = XGBClassifier(
    objective='multi:softprob', 
    random_state=42, 
    use_label_encoder=False, 
    eval_metric='mlogloss'
)

# 2. Definiamo la griglia degli iperparametri per XGBoost
# Questa selezione è ottimizzata per trovare un buon bilanciamento senza andare in overfitting
param_grid = {
    'n_estimators': [100, 150,  200],           # Numero di alberi sequenziali
    'max_depth': [3, 5, 7],               # Profondità degli alberi (XGBoost preferisce alberi più bassi di RF)
    'learning_rate': [0.01, 0.05, 0.1],    # Passo di apprendimento (regola l'impatto di ogni nuovo albero)
    'subsample': [0.6 ,0.8],              # Percentuale di dati da campionare per ogni albero
    'colsample_bytree': [0,4, 0.7]        # Percentuale di feature da campionare per ogni albero
}

# 3. Configuriama la Grid Search con 5-Fold Cross-Validation
grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=5,                                 # Sdoppia il train in 5 parti per convalidare i punteggi
    scoring='accuracy',                   # Puoi usare 'f1_macro' se le classi sono sbilanciate
    n_jobs=-1,                            # Sfrutta tutti i core della CPU in parallelo
    verbose=2                             # Mostra i log dell'avanzamento dei calcoli
)

# 4. Avviamo l'addestramento sui dati
# NOTA: Assicurati che X_train contenga solo colonne numeriche (dopo i tuoi passaggi di Target Encoding)
grid_search_xgb.fit(X_train, y_train)


# ==============================================================================
# STAMPA DEI RISULTATI
# ==============================================================================
print("\n--- Risultati della Grid Search con XGBoost ---")
print(f"Miglior Accuratezza in Cross-Validation: {grid_search_xgb.best_score_:.4f}")
print("Migliori Iperparametri Trovati:")
for param, value in grid_search_xgb.best_params_.items():
    print(f"  -> {param}: {value}")

# 5. Estraiamo il modello migliore e già pronto
best_xgb_model = grid_search_xgb.best_estimator_

Fitting 5 folds for each of 162 candidates, totalling 810 fits


c:\Users\Anna\Desktop\progetto con Gabriele\AI_Impact_on_Students\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
270 fits failed out of a total of 810.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
270 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Anna\Desktop\progetto con Gabriele\AI_Impact_on_Students\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Anna\Desktop\progetto con Gabriele\AI_Impact_on_Students\.venv\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\Anna\De


--- Risultati della Grid Search con XGBoost ---
Miglior Accuratezza in Cross-Validation: 0.5384
Migliori Iperparametri Trovati:
  -> colsample_bytree: 0.7
  -> learning_rate: 0.1
  -> max_depth: 3
  -> n_estimators: 100
  -> subsample: 0.6


In [ ]:
lista = y_test.unique()
print(lista)  # l'ordine è corretto !! (controllato)

[1 0 2]


In [21]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
best_xgb = XGBClassifier(
    objective='multi:softprob',     # Specifica che è un problema multi-classe probabilistico
    eval_metric='mlogloss',         # Metrica di valutazione interna
    colsample_bytree=0.7,           # Iperparametri ottimi trovati dalla tua Grid Search
    learning_rate=0.1,
    max_depth=3,
    n_estimators=100,
    subsample=0.6,
    random_state=42,                # Per rendere i risultati riproducibili
    n_jobs=-1                       # Sfrutta al massimo la CPU
)

best_xgb.fit(X_train, y_train)

y_pred_xgb = best_xgb.predict(X_test)

# Definizione dei nomi delle classi per rendere il report auto-esplicativo
target_names = ['Low (0)', 'Medium (1)', 'High (2)']

print(f"Accuratezza finale di XGBoost sul Test Set: {accuracy_score(y_test, y_pred_xgb):.4f}\n")
print("--- REPORT DI CLASSIFICAZIONE COMPLETO (XGBOOST) ---")
print(classification_report(y_test, y_pred_xgb, target_names=target_names))

Accuratezza finale di XGBoost sul Test Set: 0.5320

--- REPORT DI CLASSIFICAZIONE COMPLETO (XGBOOST) ---
              precision    recall  f1-score   support

     Low (0)       0.54      0.48      0.51      3274
  Medium (1)       0.48      0.61      0.54      4229
    High (2)       0.66      0.47      0.55      2497

    accuracy                           0.53     10000
   macro avg       0.56      0.52      0.53     10000
weighted avg       0.55      0.53      0.53     10000



--- Report di Classificazione Completo (random forest) ---
              precision    recall  f1-score   support

           0       0.54      0.47      0.50      3274
           1       0.48      0.62      0.54      4229
           2       0.67      0.45      0.54      2497

    accuracy                           0.53     10000
   macro avg       0.56      0.52      0.53     10000
weighted avg       0.55      0.53      0.53     10000